# Lab 3 — Hallucination (EP02)

Lab 2 ตรวจได้ว่า **รูปแบบ** ถูก แต่ถ้า model **แต่งข้อเท็จจริง** ขึ้นมาล่ะ?

**สิ่งที่จะได้เรียน**
1. **consistency sampling** — ถามซ้ำ 5 รอบ ดูว่าตอบตรงกันไหม
2. **log-prob** — ความน่าจะเป็นเฉลี่ยของ token ในคำตอบ (ต่อจาก EP01)
3. ข้อจำกัด: สองวิธีนี้จับ **ความไม่มั่นใจ** ไม่ใช่ **ความผิด** — ตรวจด้วยโค้ดให้เห็นกับตา
4. **grounding** — ตอบจากเอกสารเท่านั้น พร้อม `quote` ที่โค้ดตรวจได้ว่ามีอยู่จริง

paper ชื่อ *Temporal Patch Attention for Thai Traffic Forecasting* และประมวลรายวิชาใน lab นี้ **เราแต่งขึ้นเพื่อการสอน**
ส่วนคำตอบของ model ทั้งหมดมาจากการรันจริง

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.16 — CC BY-SA 4.0

## 0. เตรียมเครื่องก่อนเริ่ม

เหมือน Lab 1: Python 3.10+ หรือ Colab และ `OPENROUTER_API_KEY` ใน Colab Secrets หรือไฟล์ `.env`
(วิธีตั้งค่า key และเหตุผลที่เลือก model กับ provider อยู่ใน Lab 1)

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pydantic>=2.7"

In [ ]:
import os
from collections import Counter
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
PROVIDER = {"order": ["parasail/bf16"], "allow_fallbacks": False}
EXTRA = {"reasoning": {"enabled": False}, "provider": PROVIDER}

## 1. ถามซ้ำหลายรอบ + ความน่าจะเป็นของ token

ถามคำถามเดิม 5 รอบที่ `temperature=1.0` (ให้ model สุ่มได้จริง) และขอ `logprobs` มาด้วย
- **ตรงกันกี่รอบ** — ถ้าไม่ตรงกัน แปลว่า model ไม่แน่ใจ
- **log-prob เฉลี่ย** — ใกล้ 0 = มั่นใจ · ติดลบมาก = ไม่มั่นใจ

In [ ]:
def sample(question, n=5):
    answers, logprobs = [], []
    for _ in range(n):
        resp = client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": question}],
            max_tokens=30, temperature=1.0, logprobs=True, extra_body=EXTRA)
        answers.append(resp.choices[0].message.content.strip().replace(",", ""))
        tokens = resp.choices[0].logprobs.content
        logprobs.append(sum(t.logprob for t in tokens) / len(tokens))
    return answers, sum(logprobs) / n

In [ ]:
def check(question, n=5):
    answers, avg_lp = sample(question, n)
    top, count = Counter(answers).most_common(1)[0]
    flag = "⚠️ ไม่แน่ใจ" if count < n * 0.8 or avg_lp < -0.1 else "มั่นใจ"
    print(f"ตรงกัน {count}/{n} · logprob {avg_lp:.2f} · {flag} · {answers}")
    return top

## 2. ลองกับสี่คำถาม

| คำถาม | คำตอบที่ถูก |
|---|---|
| เมืองหลวงของออสเตรเลีย | แคนเบอร์รา |
| 4738 × 6219 | Python คำนวณให้ |
| 7851 × 2144 | Python คำนวณให้ |
| paper *Temporal Patch Attention for Thai Traffic Forecasting* ตีพิมพ์ที่ไหน | **ไม่มี paper นี้** — เราแต่งชื่อขึ้นเอง |

In [ ]:
capital = check("เมืองหลวงของออสเตรเลียคืออะไร ตอบแค่ชื่อเมือง")

In [ ]:
m1 = check("4738 × 6219 เท่ากับเท่าไร ตอบแค่ตัวเลข")

In [ ]:
m2 = check("7851 × 2144 เท่ากับเท่าไร ตอบแค่ตัวเลข")
paper = check("paper ชื่อ 'Temporal Patch Attention for Thai Traffic Forecasting' ตีพิมพ์ที่งานประชุมไหน ตอบแค่ชื่อย่องานประชุมกับปี")

## 3. ข้อจำกัด: มั่นใจ ≠ ถูก

การคูณตรวจด้วย Python ได้ ส่วน paper — เราแต่งชื่อขึ้นเอง จึงไม่มีงานประชุมไหนตีพิมพ์

In [ ]:
for a, b, said in [(4738, 6219, m1), (7851, 2144, m2)]:
    print(f"{a} × {b} = {a * b} · model ตอบ {said} ·", "✓" if said == str(a * b) else "✗")

## 4. Grounding: ตอบจากเอกสาร พร้อมหลักฐานที่โค้ดตรวจได้

ตำราแนะนำให้ใช้คู่กับแหล่งข้อมูลภายนอก — เราแนบ **ประมวลรายวิชา** ไปกับคำถาม สั่งให้ตอบจากเอกสารเท่านั้น
และให้ยก `quote` (ประโยคจากเอกสาร) **ก่อน** ตอบ แล้วโค้ดของเราตรวจว่า quote อยู่ในเอกสารจริง

In [ ]:
SYLLABUS = """ประมวลรายวิชา AI301 Agentic AI · ภาคเรียนที่ 2/2569
- สอบย่อยครั้งที่ 1: วันพุธที่ 18 พฤศจิกายน 2569 (บทที่ 1–2)
- สอบย่อยครั้งที่ 2: วันพุธที่ 9 ธันวาคม 2569 (บทที่ 3–4)
- ส่งโปรเจกต์: วันศุกร์ที่ 15 มกราคม 2570 ก่อน 23:59 น.
- ส่งงานช้า หักวันละ 10% ไม่เกิน 3 วัน"""

In [ ]:
from pydantic import BaseModel, ConfigDict, Field

class Grounded(BaseModel):
    model_config = ConfigDict(extra="forbid")
    found: bool = Field(description="เอกสารมีข้อมูลที่ตอบคำถามนี้หรือไม่")
    quote: str = Field(description="บรรทัดจากเอกสารที่ใช้ตอบ คัดลอกตรงตัวทั้งบรรทัด หรือ '' ถ้า found เป็น false")
    answer: str = Field(description="คำตอบสั้นๆ หรือ 'ไม่ทราบ' ถ้า found เป็น false")

In [ ]:
GROUND_SYSTEM = ("คุณคือผู้ช่วยอาจารย์ ตอบจากเอกสารใน <doc> เท่านั้น ถ้าเอกสารไม่ได้บอก ให้ตอบว่า 'ไม่ทราบ' ห้ามเดา "
                 "quote ต้องคัดลอกจากเอกสารตรงตัวทั้งบรรทัด")
SCHEMA = {"type": "json_schema",
          "json_schema": {"name": "Grounded", "strict": True, "schema": Grounded.model_json_schema()}}

def ask_doc(question):
    msgs = [{"role": "system", "content": GROUND_SYSTEM},
            {"role": "user", "content": f"<doc>\n{SYLLABUS}\n</doc>\n\nคำถาม: {question}"}]
    resp = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=200, temperature=0,
                                          response_format=SCHEMA, extra_body=EXTRA)
    return Grounded.model_validate_json(resp.choices[0].message.content)

In [ ]:
for q in ["สอบย่อยครั้งที่ 3 วันไหนครับ", "สอบย่อยครั้งที่ 2 ออกบทไหนบ้างครับ", "ส่งงานช้าได้ไม่เกินกี่วันครับ"]:
    g = ask_doc(q)
    print(q, "→", g.answer)
    if not g.found:
        print("   เอกสารไม่ได้บอก → ไม่ตอบเอง")
    elif g.quote in SYLLABUS:
        print("   ✓ quote อยู่ในเอกสารจริง:", g.quote)
    else:
        print("   ✗ quote ไม่อยู่ในเอกสาร:", g.quote)

## สรุป

| คำถาม | ตรงกัน · logprob | ป้ายของเรา | จริง |
|---|---|---|---|
| เมืองหลวงของออสเตรเลีย | 5/5 · ≈ 0 | มั่นใจ | ✓ |
| 4738 × 6219 | แทบไม่ซ้ำ · ติดลบชัด | ⚠️ ไม่แน่ใจ | ✗ |
| 7851 × 2144 | เลขเดิมเกือบทุกรอบ · ≈ 0 | มั่นใจ | ✗ มั่นใจแต่ผิด |
| paper ที่ไม่มีอยู่จริง | "AAAI 2024" เกือบทุกรอบ · ≈ 0 | มั่นใจ | ✗ ไม่มี paper นี้ |
| grounding + quote | ตอบ "ไม่ทราบ" เมื่อเอกสารไม่ได้บอก · quote ตรวจได้ | | ✓ |

**ข้อคิดสำหรับ agent:** สิ่งที่ตรวจด้วยโค้ดได้ (เลขคณิต, quote ในเอกสาร) ให้ตรวจด้วยโค้ด —
agent ที่ดีจึงเรียก **tool** (เช่นเครื่องคิดเลข หรือการค้นเอกสาร) แทนการเดา ซึ่งเราจะได้เห็นใน EP ถัดๆ ไป

**แบบฝึกหัด:** เพิ่มบรรทัด "สอบปลายภาค: …" ลงใน `SYLLABUS` แล้วถามคำถามที่ต้องใช้สองบรรทัดพร้อมกัน
เช่น "สอบย่อยครั้งที่ 2 ห่างจากสอบปลายภาคกี่วัน" — quote ยังตรวจผ่านไหม?

▶ **ถัดไป: Lab 4 — LangChain** สร้างตัวคัดแยกอีเมลจาก Lab 2 ใหม่ด้วย framework แล้วเทียบกับโค้ดที่เราเขียนเอง